# **Load Parameters**

In [0]:
import traceback

try:
    catalog = dbutils.widgets.get("catalog")
    bronze_schema = dbutils.widgets.get("bronze_schema")
    bronze_table = dbutils.widgets.get("bronze_table")
    process_path = dbutils.widgets.get("process_path")
    processed_path = dbutils.widgets.get("processed_path")

except Exception as e:
    print("Error getting notebook parameters")
    print(traceback.format_exc())
    raise e

In [0]:
# import traceback
# catalog ='iowa_sales'
# bronze_schema = 'sales_bronze'
# bronze_table = 'sales_raw'
# process_path = '/Volumes/iowa_sales/sales_bronze/process'
# processed_path = '/Volumes/iowa_sales/sales_bronze/processed'

## **Define a Raw Schema**

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, IntegerType, 
    DoubleType, DateType
)

# All columns are loaded as StringType to prevent data loss during ingestion.
raw_schema = StructType([
    StructField("invoice_line_no", StringType(), True),
    StructField("date", StringType(), True),
    StructField("store", StringType(), True),
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zipcode", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("county_number", StringType(), True),
    StructField("county", StringType(), True),
    StructField("category", StringType(), True),
    StructField("category_name", StringType(), True),
    StructField("vendor_no", StringType(), True),
    StructField("vendor_name", StringType(), True),
    StructField("itemno", StringType(), True),
    StructField("im_desc", StringType(), True),
    StructField("pack", StringType(), True),
    StructField("bottle_volume_ml", StringType(), True),
    StructField("state_bottle_cost", StringType(), True),
    StructField("state_bottle_retail", StringType(), True),
    StructField("sale_bottles", StringType(), True),
    StructField("sale_dollars", StringType(), True),
    StructField("sale_liters", StringType(), True),
    StructField("sale_gallons", StringType(), True),
    StructField(":@computed_region_3r5t_5243", StringType(), True),
    StructField(":@computed_region_wnea_7qqw", StringType(), True),
    StructField(":@computed_region_i9mz_6gmt", StringType(), True),
    StructField(":@computed_region_uhgg_e8y2", StringType(), True),
    StructField(":@computed_region_e7ym_nrbf", StringType(), True)
])

## **Load Raw Data from Volume**

In [0]:
import os
from pyspark.sql import DataFrame

try:
    files = dbutils.fs.ls(process_path)
    csv_files = [f.path for f in files if f.path.endswith('.csv')]

    if not csv_files:
        # Exit the notebook gracefully if no files are present
        dbutils.notebook.exit("Files Not Found")
    
    # Store matched files in a variable for later use in archiving
    matched_files = csv_files
    
    # Read all matched CSV files into a single DataFrame
    df_raw_sales = (spark.read
        .schema(raw_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE") # 'PERMISSIVE' keeps malformed rows, which we handle later
        .csv(matched_files)
    )

    file_names = [os.path.basename(f) for f in matched_files]
    print(f"Loaded Files: {', '.join(file_names)}")

except Exception as e:
    print(f"Error loading files from {process_path}")
    print(traceback.format_exc())
    # dbutils.notebook.exit(f"Error loading files: {str(e)}")

## **Create Bronze Table**

In [0]:
try:
    # Get column names and types from the DataFrame schema
    columns = [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in df_raw_sales.schema.fields
    ]
    
    # Add audit columns to the table definition
    columns.append("saved_date TIMESTAMP")
    columns.append("is_corrupted BOOLEAN")

    # Define the CREATE TABLE query
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{bronze_schema}.{bronze_table} (
      {', '.join(columns)}
    )
    USING DELTA
    """
    
    print(f"Creating or verifying table {catalog}.{bronze_schema}.{bronze_table}...")
    spark.sql(create_table_query)

except Exception as e:
    print("Error creating Bronze table")
    print(traceback.format_exc())
    raise e

## **Ingest Data into Bronze Table**

In [0]:
from pyspark.sql.functions import current_timestamp

try:
    # Add the audit timestamp column
    df_raw_sales_stagged = df_raw_sales.withColumn("saved_date", current_timestamp())
    
    # Create a temporary view to be queried by SQL
    df_raw_sales_stagged.createOrReplaceTempView("sales_raw_staging_view")

    truncate_query = f"""TRUNCATE TABLE {catalog}.{bronze_schema}.{bronze_table}"""
    spark.sql(truncate_query)

    # Use INSERT OVERWRITE for an atomic and idempotent write
    insert_query = f"""
    INSERT INTO {catalog}.{bronze_schema}.{bronze_table}
    SELECT
        invoice_line_no,
        date,
        store,
        name,
        address,
        city,
        zipcode,
        store_location,
        county_number,
        county,
        category,
        category_name,
        vendor_no,
        vendor_name,
        itemno,
        im_desc,
        pack,
        bottle_volume_ml,
        state_bottle_cost,
        state_bottle_retail,
        sale_bottles,
        sale_dollars,
        sale_liters,
        sale_gallons,
        `:@computed_region_3r5t_5243`, 
        `:@computed_region_wnea_7qqw`, 
        `:@computed_region_i9mz_6gmt`, 
        `:@computed_region_uhgg_e8y2`, 
        `:@computed_region_e7ym_nrbf`, 
        saved_date,
        
        -- Data quality check: Flag rows as 'corrupted' if key fields
        -- cannot be cast to their proper numeric types.
        (
            -- Integer column checks
            (store IS NOT NULL AND try_cast(store AS INT) IS NULL) OR
            (county_number IS NOT NULL AND try_cast(county_number AS INT) IS NULL) OR
            (category IS NOT NULL AND try_cast(category AS INT) IS NULL) OR
            (vendor_no IS NOT NULL AND try_cast(vendor_no AS INT) IS NULL) OR
            (itemno IS NOT NULL AND try_cast(itemno AS INT) IS NULL) OR
            (sale_bottles IS NOT NULL AND try_cast(sale_bottles AS INT) IS NULL) OR
            
            (pack IS NOT NULL AND try_cast(try_cast(pack AS DOUBLE) AS INT) IS NULL) OR
            (bottle_volume_ml IS NOT NULL AND try_cast(try_cast(bottle_volume_ml AS DOUBLE) AS INT) IS NULL) OR
            -- Decimal/Double column checks
            (state_bottle_cost IS NOT NULL AND try_cast(state_bottle_cost AS DOUBLE) IS NULL) OR
            (state_bottle_retail IS NOT NULL AND try_cast(state_bottle_retail AS DOUBLE) IS NULL) OR
            (sale_dollars IS NOT NULL AND try_cast(sale_dollars AS DOUBLE) IS NULL) OR
            (sale_liters IS NOT NULL AND try_cast(sale_liters AS DOUBLE) IS NULL) OR
            (sale_gallons IS NOT NULL AND try_cast(sale_gallons AS DOUBLE) IS NULL) 
        ) AS is_corrupted    
    FROM
        sales_raw_staging_view;
    """
    
    print(f"Ingesting data into {catalog}.{bronze_schema}.{bronze_table}...")
    spark.sql(insert_query)
    print("Ingestion complete.")

except Exception as e:
    print("Error during data ingestion")
    print(traceback.format_exc())
    raise e

## **Archive Processed Files**

In [0]:
from datetime import datetime
import os

# Check if the 'matched_files' variable exists and is not empty
if 'matched_files' in locals() and matched_files:
    try:
        # Get current date in YYYYMMDD format
        date_suffix = datetime.now().strftime("%Y%m%d")
        print(f"Archiving {len(matched_files)} processed files...")

        for f in matched_files:
            base_name = os.path.basename(f)
            name, ext = os.path.splitext(base_name)
            
            # Add date suffix to the file name
            new_name = f"{name}_{date_suffix}{ext}"
            dest_path = os.path.join(processed_path, new_name)
            
            # Move the file
            dbutils.fs.mv(f, dest_path)
        
        print(f"Files successfully archived to {processed_path}")

    except Exception as e:
        print("Error during file archiving")
        print(traceback.format_exc())
        raise e
else:
    print("No files to archive (ingestion may have failed or no files were found).")

# Final exit to signal success to the job orchestrator
dbutils.notebook.exit("Bronze layer processing complete.")